# Data Cleaning Exploration for data68k.json

This notebook explores the data to understand what changes are needed based on CHANGES.md.

## Changes from CHANGES.md:
1. Delete `aider_commit`
2. Delete all `continue.dev` apart from tasks 2, 8, 13, 22
3. Delete all `irl_cursor` samples
4. Delete the first system prompt in all `honeypot` samples
5. Delete all `aider_manual` samples
6. Delete all `aider_multi` samples apart from specific ones
7. Re-label `hackaprompt`, `gandalf_ignore_instructions` and `gandalf_summarization` as evaluations
8. Delete `prosocial_dialog`
9. Delete `prism`

## Key Finding Summary:
Based on grep analysis of data68k.json:

| Benchmark | Count | Action |
|-----------|-------|--------|
| prosocial_dialog | 3000 | DELETE |
| prism | 3000 | DELETE |
| hackaprompt_dataset | 3000 | RE-LABEL (eval_category: false -> true) |
| gandalf_ignore_instructions | 777 | Already eval_category=true (no change) |
| gandalf_summarization | 114 | Already eval_category=true (no change) |
| aider_commit | 0 | Not in dataset |
| irl_cursor | 0 | Not in dataset |
| aider_manual | 0 | Not in dataset |
| aider_multi | 0 | Not in dataset |
| continue.dev | 0 | Not in dataset |
| honeypot | 0 | Not in dataset |

In [1]:
import json
from collections import Counter

try:
    import ijson
    USE_IJSON = True
except ImportError:
    USE_IJSON = False
    print("ijson not available, using fallback method")

In [2]:
# Load data
data = []

if USE_IJSON:
    with open('./data68k.json', 'rb') as f:
        try:
            for record in ijson.items(f, 'item'):
                data.append(record)
        except ijson.IncompleteJSONError:
            print(f"File truncated, but recovered {len(data)} complete records")
else:
    with open('./data68k.json', 'r') as f:
        content = f.read()
    last_complete = content.rfind('},\n  {')
    if last_complete == -1:
        last_complete = content.rfind('}\n  ]')
    if last_complete != -1:
        truncated_content = content[:last_complete+1] + '\n]'
        data = json.loads(truncated_content)

print(f"Total records loaded: {len(data)}")

Total records loaded: 68166


## 1. Overview of all benchmarks

In [3]:
# Extract benchmark names from IDs
benchmarks = []
for item in data:
    record_id = item.get('id', '')
    benchmark = record_id.split(':')[0] if ':' in record_id else record_id
    benchmarks.append(benchmark)

benchmark_counts = Counter(benchmarks)
print(f"Total unique benchmarks: {len(benchmark_counts)}")
print("\nAll benchmarks and counts:")
for name, count in sorted(benchmark_counts.items()):
    print(f"  {name}: {count}")

Total unique benchmarks: 38

All benchmarks and counts:
  author_claude: 8
  awesome_chatgpt: 203
  aya_redteaming: 2000
  categoricalharmfulqa: 550
  cdial_bias: 3000
  civics: 700
  cvalues_rlhf: 2000
  discrim_eval: 2000
  do_not_answer: 939
  ethics: 2000
  gandalf_ignore_instructions: 777
  gandalf_summarization: 114
  gest: 2000
  hackaprompt_dataset: 3000
  harmfulqa: 1924
  hh_rlhf: 3000
  kobbq: 2000
  llm_global_opinions: 2000
  mmlu: 2000
  model_written_evals: 2000
  moralexceptqa: 148
  mosscap_prompt_injection: 2000
  natural_reasoning: 2000
  nl2bash: 2000
  or_bench: 2000
  prism: 3000
  prosocial_dialog: 3000
  real_toxicity_prompts: 2000
  researchy_questions: 3000
  s_eval: 2000
  safetybench: 2000
  sharegpt: 3000
  simplesafetytests: 80
  toxic_chat: 3000
  toxigen_data: 3000
  ultrasafety: 2000
  wmdp: 1273
  xstest_v2_copy: 450


## 2. Check benchmarks from CHANGES.md

In [4]:
# Benchmarks to delete entirely
delete_entirely = [
    'aider_commit',
    'irl_cursor',
    'aider_manual',
    'prosocial_dialog',
    'prism'
]

print("Benchmarks to DELETE ENTIRELY:")
total_to_delete = 0
for bench in delete_entirely:
    count = benchmark_counts.get(bench, 0)
    total_to_delete += count
    status = "EXISTS - WILL DELETE" if count > 0 else "NOT IN DATASET"
    print(f"  {bench}: {count} samples [{status}]")

print(f"\nTotal samples to delete (entire benchmarks): {total_to_delete}")

Benchmarks to DELETE ENTIRELY:
  aider_commit: 0 samples [NOT IN DATASET]
  irl_cursor: 0 samples [NOT IN DATASET]
  aider_manual: 0 samples [NOT IN DATASET]
  prosocial_dialog: 3000 samples [EXISTS - WILL DELETE]
  prism: 3000 samples [EXISTS - WILL DELETE]

Total samples to delete (entire benchmarks): 6000


In [5]:
# Check partial deletion benchmarks
partial_delete = ['continue.dev', 'aider_multi']

print("Benchmarks requiring PARTIAL DELETION:")
for bench in partial_delete:
    count = benchmark_counts.get(bench, 0)
    status = "EXISTS - NEEDS FILTERING" if count > 0 else "NOT IN DATASET"
    print(f"  {bench}: {count} samples [{status}]")

Benchmarks requiring PARTIAL DELETION:
  continue.dev: 0 samples [NOT IN DATASET]
  aider_multi: 0 samples [NOT IN DATASET]


In [6]:
# Check honeypot (needs system prompt removal)
honeypot_count = benchmark_counts.get('honeypot', 0)
print(f"\nHoneypot samples (need system prompt removal):")
print(f"  honeypot: {honeypot_count} samples [{'EXISTS' if honeypot_count > 0 else 'NOT IN DATASET'}]")


Honeypot samples (need system prompt removal):
  honeypot: 0 samples [NOT IN DATASET]


In [7]:
# Check benchmarks to re-label as evaluations
relabel_benchmarks = ['hackaprompt_dataset', 'gandalf_ignore_instructions', 'gandalf_summarization']

print("Benchmarks to RE-LABEL as evaluations:")
for bench_name in relabel_benchmarks:
    samples = [item for item in data if item.get('id', '').startswith(bench_name)]
    if samples:
        eval_cats = Counter(item.get('metadata', {}).get('eval_category') for item in samples)
        print(f"\n{bench_name}:")
        print(f"  Total samples: {len(samples)}")
        print(f"  Current eval_category values: {dict(eval_cats)}")
        if eval_cats.get(False, 0) > 0:
            print(f"  ACTION: Need to re-label {eval_cats.get(False, 0)} samples from False to True")
        else:
            print(f"  No action needed - all samples already have eval_category=True")

Benchmarks to RE-LABEL as evaluations:

hackaprompt_dataset:
  Total samples: 3000
  Current eval_category values: {False: 3000}
  ACTION: Need to re-label 3000 samples from False to True

gandalf_ignore_instructions:
  Total samples: 777
  Current eval_category values: {True: 777}
  No action needed - all samples already have eval_category=True

gandalf_summarization:
  Total samples: 114
  Current eval_category values: {True: 114}
  No action needed - all samples already have eval_category=True


## 3. Look for other potential data issues

In [8]:
# Check for empty/short inputs
empty_inputs = []
short_inputs = []  # Less than 2 messages

for item in data:
    input_msgs = item.get('input', [])
    if not input_msgs:
        empty_inputs.append(item.get('id'))
    elif len(input_msgs) < 2:
        short_inputs.append((item.get('id'), len(input_msgs)))

print(f"Empty inputs: {len(empty_inputs)}")
if empty_inputs:
    print(f"  Examples: {empty_inputs[:5]}")

print(f"\nVery short inputs (<2 messages): {len(short_inputs)}")
if short_inputs:
    print(f"  Examples: {short_inputs[:10]}")

Empty inputs: 0

Very short inputs (<2 messages): 0


In [9]:
# Check for empty content in messages
empty_content = []
for item in data:
    item_id = item.get('id', 'unknown')
    input_msgs = item.get('input', [])
    for i, msg in enumerate(input_msgs):
        content = msg.get('content', '')
        if isinstance(content, str) and len(content) == 0:
            empty_content.append((item_id, i, msg.get('role')))

print(f"Messages with empty content: {len(empty_content)}")
if empty_content:
    empty_by_bench = Counter()
    for id_, i, role in empty_content:
        bench = id_.split(':')[0] if ':' in id_ else id_
        empty_by_bench[bench] += 1
    print("By benchmark:")
    for bench, count in empty_by_bench.most_common(10):
        print(f"  {bench}: {count}")
    print("\nExamples:")
    for item in empty_content[:5]:
        print(f"  {item}")

Messages with empty content: 17
By benchmark:
  sharegpt: 14
  hh_rlhf: 3

Examples:
  ('hh_rlhf:106448', 2, 'assistant')
  ('hh_rlhf:103301', 2, 'assistant')
  ('sharegpt:4fqz138_55', 7, 'user')
  ('sharegpt:dnuuvkc_0', 5, 'user')
  ('hh_rlhf:9858', 2, 'assistant')


In [10]:
# Check for duplicate IDs
id_counts = Counter(item.get('id') for item in data)
duplicates = {id_: count for id_, count in id_counts.items() if count > 1}

print(f"Duplicate IDs: {len(duplicates)}")
if duplicates:
    print("\nDuplicates:")
    for id_, count in sorted(duplicates.items(), key=lambda x: -x[1])[:20]:
        print(f"  {id_}: {count} occurrences")

Duplicate IDs: 0


In [11]:
# Check for malformed messages
malformed_messages = []

for item in data:
    item_id = item.get('id', 'unknown')
    input_msgs = item.get('input', [])
    for i, msg in enumerate(input_msgs):
        if not isinstance(msg, dict):
            malformed_messages.append((item_id, i, 'not a dict', type(msg)))
        elif 'role' not in msg:
            malformed_messages.append((item_id, i, 'missing role', list(msg.keys())))
        elif 'content' not in msg:
            malformed_messages.append((item_id, i, 'missing content', list(msg.keys())))

print(f"Malformed messages: {len(malformed_messages)}")
if malformed_messages:
    print("Examples:")
    for item in malformed_messages[:10]:
        print(f"  {item}")

Malformed messages: 0


## 4. Summary of changes needed

In [12]:
print("=" * 70)
print("SUMMARY OF CLEANING ACTIONS")
print("=" * 70)

# Delete entire benchmarks
delete_count = sum(benchmark_counts.get(b, 0) for b in ['prosocial_dialog', 'prism'])
print(f"\n1. DELETE ENTIRE BENCHMARKS: {delete_count} samples")
print(f"   - prosocial_dialog: {benchmark_counts.get('prosocial_dialog', 0)}")
print(f"   - prism: {benchmark_counts.get('prism', 0)}")

# Re-label
hackaprompt_false = sum(1 for item in data 
                        if item.get('id', '').startswith('hackaprompt_dataset') 
                        and item.get('metadata', {}).get('eval_category') == False)
print(f"\n2. RE-LABEL AS EVALUATIONS: {hackaprompt_false} samples")
print(f"   - hackaprompt_dataset: {hackaprompt_false} (set eval_category=True)")
print(f"   - gandalf_ignore_instructions: 0 (already correct)")
print(f"   - gandalf_summarization: 0 (already correct)")

# Not in dataset
print(f"\n3. NOT IN DATASET (no action needed):")
for bench in ['aider_commit', 'irl_cursor', 'aider_manual', 'continue.dev', 'aider_multi', 'honeypot']:
    print(f"   - {bench}")

# Data quality issues
print(f"\n4. DATA QUALITY ISSUES (optional fixes):")
print(f"   - Messages with empty content: {len(empty_content)}")
print(f"   - Duplicate IDs: {len(duplicates)}")
print(f"   - Malformed messages: {len(malformed_messages)}")

# Final stats
final_count = len(data) - delete_count
print(f"\n" + "=" * 70)
print(f"EXPECTED RESULT:")
print(f"  Original records: {len(data)}")
print(f"  After cleaning: {final_count}")
print(f"  Total removed: {delete_count}")
print(f"  Re-labeled: {hackaprompt_false}")
print("=" * 70)

SUMMARY OF CLEANING ACTIONS

1. DELETE ENTIRE BENCHMARKS: 6000 samples
   - prosocial_dialog: 3000
   - prism: 3000

2. RE-LABEL AS EVALUATIONS: 3000 samples
   - hackaprompt_dataset: 3000 (set eval_category=True)
   - gandalf_ignore_instructions: 0 (already correct)
   - gandalf_summarization: 0 (already correct)

3. NOT IN DATASET (no action needed):
   - aider_commit
   - irl_cursor
   - aider_manual
   - continue.dev
   - aider_multi
   - honeypot

4. DATA QUALITY ISSUES (optional fixes):
   - Messages with empty content: 17
   - Duplicate IDs: 0
   - Malformed messages: 0

EXPECTED RESULT:
  Original records: 68166
  After cleaning: 62166
  Total removed: 6000
  Re-labeled: 3000


---

# Data Cleaning Code

The following code will perform the cleaning operations.

In [13]:
def clean_data(data):
    """
    Clean the dataset according to CHANGES.md specifications.
    
    For data68k.json, the following changes apply:
    1. Delete prosocial_dialog (3000 samples)
    2. Delete prism (3000 samples)
    3. Re-label hackaprompt_dataset as eval_category=True
    
    Returns:
        cleaned_data: list of cleaned records
        stats: dict with cleaning statistics
    """
    stats = {
        'original_count': len(data),
        'deleted_prosocial_dialog': 0,
        'deleted_prism': 0,
        'relabeled_hackaprompt': 0,
    }
    
    # Benchmarks to delete entirely
    delete_entirely = {'prosocial_dialog', 'prism'}
    
    # Benchmarks to re-label as evaluations
    relabel_as_eval = {'hackaprompt_dataset'}
    
    cleaned_data = []
    
    for item in data:
        item_id = item.get('id', '')
        benchmark = item_id.split(':')[0] if ':' in item_id else item_id
        
        # 1. Skip if benchmark should be deleted entirely
        if benchmark == 'prosocial_dialog':
            stats['deleted_prosocial_dialog'] += 1
            continue
        if benchmark == 'prism':
            stats['deleted_prism'] += 1
            continue
        
        # Make a copy to avoid modifying original
        item = dict(item)
        
        # 2. Re-label hackaprompt_dataset as evaluations
        if benchmark in relabel_as_eval:
            if 'metadata' not in item:
                item['metadata'] = {}
            else:
                item['metadata'] = dict(item['metadata'])  # Copy metadata
            
            if item['metadata'].get('eval_category') != True:
                item['metadata']['eval_category'] = True
                item['metadata']['eval_type'] = 'alignment'  # hackaprompt is alignment eval
                stats['relabeled_hackaprompt'] += 1
        
        cleaned_data.append(item)
    
    stats['final_count'] = len(cleaned_data)
    stats['total_deleted'] = stats['original_count'] - stats['final_count']
    
    return cleaned_data, stats

In [14]:
# DRY RUN - Preview what the cleaning would do
cleaned_data, stats = clean_data(data)

print("DRY RUN - Cleaning Statistics:")
print("=" * 50)
print(f"Original count: {stats['original_count']}")
print(f"Final count: {stats['final_count']}")
print(f"Total deleted: {stats['total_deleted']}")
print()
print("Breakdown:")
print(f"  Deleted prosocial_dialog: {stats['deleted_prosocial_dialog']}")
print(f"  Deleted prism: {stats['deleted_prism']}")
print(f"  Re-labeled hackaprompt_dataset: {stats['relabeled_hackaprompt']}")

DRY RUN - Cleaning Statistics:
Original count: 68166
Final count: 62166
Total deleted: 6000

Breakdown:
  Deleted prosocial_dialog: 3000
  Deleted prism: 3000
  Re-labeled hackaprompt_dataset: 3000


In [15]:
# Verify cleaned data
print("Verification - Benchmarks in cleaned data:")
cleaned_benchmarks = Counter()
for item in cleaned_data:
    item_id = item.get('id', '')
    benchmark = item_id.split(':')[0] if ':' in item_id else item_id
    cleaned_benchmarks[benchmark] += 1

print(f"\nTotal benchmarks: {len(cleaned_benchmarks)}")
for name, count in sorted(cleaned_benchmarks.items()):
    print(f"  {name}: {count}")

# Verify deletions
print("\n" + "="*50)
print("Verification - Deleted benchmarks should be 0:")
print(f"  prosocial_dialog: {cleaned_benchmarks.get('prosocial_dialog', 0)}")
print(f"  prism: {cleaned_benchmarks.get('prism', 0)}")

# Verify re-labeling
print("\nVerification - hackaprompt_dataset eval_category values:")
hackaprompt_samples = [item for item in cleaned_data if item.get('id', '').startswith('hackaprompt_dataset')]
eval_cats = Counter(item.get('metadata', {}).get('eval_category') for item in hackaprompt_samples)
print(f"  {dict(eval_cats)}")

Verification - Benchmarks in cleaned data:

Total benchmarks: 36
  author_claude: 8
  awesome_chatgpt: 203
  aya_redteaming: 2000
  categoricalharmfulqa: 550
  cdial_bias: 3000
  civics: 700
  cvalues_rlhf: 2000
  discrim_eval: 2000
  do_not_answer: 939
  ethics: 2000
  gandalf_ignore_instructions: 777
  gandalf_summarization: 114
  gest: 2000
  hackaprompt_dataset: 3000
  harmfulqa: 1924
  hh_rlhf: 3000
  kobbq: 2000
  llm_global_opinions: 2000
  mmlu: 2000
  model_written_evals: 2000
  moralexceptqa: 148
  mosscap_prompt_injection: 2000
  natural_reasoning: 2000
  nl2bash: 2000
  or_bench: 2000
  real_toxicity_prompts: 2000
  researchy_questions: 3000
  s_eval: 2000
  safetybench: 2000
  sharegpt: 3000
  simplesafetytests: 80
  toxic_chat: 3000
  toxigen_data: 3000
  ultrasafety: 2000
  wmdp: 1273
  xstest_v2_copy: 450

Verification - Deleted benchmarks should be 0:
  prosocial_dialog: 0
  prism: 0

Verification - hackaprompt_dataset eval_category values:
  {True: 3000}


In [18]:
# FINAL STEP: Save cleaned data
from decimal import Decimal

class DecimalEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return float(obj)
        return super().default(obj)

output_path = './data68k_cleaned.json'
with open(output_path, 'w') as f:
    json.dump(cleaned_data, f, indent=2, cls=DecimalEncoder)
print(f"Saved cleaned data to {output_path}")
print(f"Total records: {len(cleaned_data)}")

Saved cleaned data to ./data68k_cleaned.json
Total records: 62166
